## **Контест 1**

### Введение

**Задача:** Построить модель для прогнозирования стоимости поездки на самокате по данным о прошлых поездках, погоде и спросе.

**Данные:**
- train.csv — обучающая выборка с признаками поездок и целевой переменной (price).
- test.csv — тестовая выборка.
- sample_submission.csv — шаблон для отправки результатов.

В данных присутствуют числовые и категориальные признаки, а также дополнительные признаки, созданные вручную для повышения качества модели.


### Эксперименты

#### Подбор дополнительных фич

Для простоты и получения быстрых результатов были использованы 2 модели:

##### Linear Regressoin

```python
linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)
linear_y_pred = linear_model.predict(X=X_test_encoded)

r2 = r2_score(y_test, linear_y_pred)
r2
```
_Прогресс_
1. База: **0.8123655530184463**
2. Добавлены фичи: **0.8468610667435832**

##### Random Forest Regression 
```python
rf_model = RandomForestRegressor(n_estimators=200, max_depth=50, n_jobs=-1)

param_grid = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [5, 8, 10, 12]
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='r2')
grid_search.fit(X_train_encoded, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучший R^2 на кросс-валидации: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_encoded)

r2 = r2_score(y_test, y_pred)
r2
```
_Прогресс_
1. База: **0.8362440952207835**
2. Добавлены фичи: **0.84218578304564**


#### Подбор моделей

Но для данных моделей потолок был - 0.84, поэтому было принято решение использовать стекинг моделей.

1. `Ridge` - регуляризованная линейная модель. Обычная линейная регрессия не справляется с нелинейными зависимостями и чувствительна к мультиколлинеарности. Ridge решает эти проблемы за счёт регуляризации.
2. `CatBoost` - градиентный бустинг, который хорошо работает с категориальными признаками и требует минимальной предобработки, устойчив к переобучению. В отличие от LightGBM, проще в настройке и показал лучший результат, поэтому был выбран в итоговом варианте.
3. `GradientBoostingRegressor` - классический бустинг по решающим деревьям, который хорошо выявляет сложные нелинейные зависимости. Он стабилен, легко настраивается и даёт сильную базу для ансамбля.

Этот стек сочетает разные подходы.
`Ridge` стабилизирует ансамбль и хорошо работает с мультиколлинеарностью.
`CatBoost` отлично работает с категориальными признаками и минимально требует ручной обработки данных.
`GradientBoostingRegressor` добавляет устойчивость и разнообразие, что снижает риск переобучения и повышает итоговую метрику.

_После смены подхода с моделями выбранные фичи снова были проверены повторно, но добавление/удаление не дало улучшений, поэтому они были выбраны в итоговом варианте._

_Импорты_

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor
import numpy as np
import time

_загрузка данных_

In [4]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.head()

,id,trip_duration_min,distance_km,battery_level_start,temperature_c,wind_speed,demand_index,distance_km_noisy,city_zone,scooter_model,is_weekend,rental_price,avg_price_last_week,route_complexity,driver_experience,weather_rating
0,0,25.599707,7.441135,78.519717,6.128784,8.680982,0.861114,11.102968,center,C,0,16.238355,4.985755,-0.777649,6.163800,3
1,1,57.289287,8.770495,NaN,15.947497,8.812507,0.078338,9.426186,park,A,1,6.231436,3.751414,1.587773,2.568192,3
2,2,45.259667,6.147492,31.714901,26.099837,3.501865,0.072575,6.285575,center,A,0,8.914843,2.666093,2.251650,5.311526,1
3,3,37.926217,7.740660,86.634377,11.560209,NaN,0.850318,8.455020,park,B,1,11.197729,5.835958,0.567339,9.418034,2
4,4,13.581025,3.846835,63.223723,15.542815,7.245579,0.212850,7.641726,suburb,B,0,0.560182,0.058212,1.688476,5.726172,1


In [5]:
print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')

display(train.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_train'))
display(test.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_test'))

display(
    train['rental_price']
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .to_frame()
    .T
)

num_cols = [c for c in train.select_dtypes(include=np.number).columns if c != 'rental_price']
corr = train[num_cols + ['rental_price']].corr(numeric_only=True)['rental_price'].sort_values(ascending=False)
display(corr.head(10).to_frame('corr_with_target'))

train shape: (1200, 16)
test shape: (400, 15)


,missing_train
route_complexity,115
battery_level_start,107
wind_speed,101
temperature_c,87
distance_km,0
trip_duration_min,0
demand_index,0
id,0
distance_km_noisy,0
city_zone,0


,missing_test
wind_speed,43
battery_level_start,42
route_complexity,38
temperature_c,35
distance_km,0
id,0
trip_duration_min,0
demand_index,0
distance_km_noisy,0
scooter_model,0


,count,mean,std,min,1%,5%,50%,95%,99%,max
rental_price,1200.0,8.750015,5.79211,0.5,0.5,1.446607,7.485541,20.742103,27.236083,35.81399


,corr_with_target
rental_price,1.000000
avg_price_last_week,0.854657
distance_km,0.612479
distance_km_noisy,0.578063
trip_duration_min,0.543025
demand_index,0.423578
is_weekend,0.052288
temperature_c,0.029204
driver_experience,0.027401
route_complexity,0.024169


#### Дополнительные фичи

- `speed` — средняя скорость. Помогает различать короткие быстрые и длинные медленные поездки.

- `distance_diff` — абсолютная разница между чистым и шумным расстоянием. Говорит о качестве измерений. <span style="color: red;">Пока под вопросом</span>

- `demand` / `demand_log` / `demand_bin` — индекс спроса (клиппинг + лог + бины).

- `demand_by_distance`, `demand_by_duration`, `demand_weekend` — взаимодействия спроса с дистанцией/длительностью/выходными, ловят нелинейные эффекты.

- Биннинг `trip_duration` / `distance` — категории короткая/средняя/длинная. Полезно при разных режимах тарифов/поведения.


In [6]:
def make_features(df):
    df = df.copy()

    df['speed'] = df['distance_km'] / (df['trip_duration_min'].replace(0, 1) / 60)
    df['distance_diff'] = (df['distance_km'] - df['distance_km_noisy']).abs()

    df['demand'] = df['demand_index'].astype(float)

    df['demand_clip'] = df['demand'].clip(0, df['demand'].quantile(0.99))
    df['demand_log'] = np.log1p(df['demand_clip'])

    q = df['demand'].quantile([0.33, 0.66]).values
    df['demand_bin'] = pd.cut(df['demand'], bins=[-np.inf, q[0], q[1], np.inf], labels=[0,1,2]).astype(int)

    df['demand_by_distance'] = df['demand'] * df['distance_km']
    df['demand_by_duration'] = df['demand'] * df['trip_duration_min']
    df['demand_weekend'] = df['demand'] * df['is_weekend']

    return df

_Предобработка данных_

In [7]:
train_f = make_features(train)
test_f = make_features(test)

y = train_f['rental_price'].values
X = train_f.drop(columns=['rental_price', 'id']).copy()
X_sub = test_f.drop(columns=['id']).copy()

### **Крутые модели**

In [8]:

start_time = time.time()

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

key_poly = ['avg_price_last_week', 'distance_km', 'trip_duration_min', 'demand_index']
rest_num = [c for c in num_cols if c not in key_poly]

pre = ColumnTransformer([
    ('poly_num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler())
    ]), key_poly),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), rest_num),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
], remainder='drop')

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_ridge = np.zeros(len(X))
oof_gbr = np.zeros(len(X))
oof_cat = np.zeros(len(X))

pred_ridge = np.zeros(len(X_sub))
pred_gbr = np.zeros(len(X_sub))
pred_cat = np.zeros(len(X_sub))

X_cat_full = X.copy()
X_sub_cat = X_sub.copy()
for c in cat_cols:
    X_cat_full[c] = X_cat_full[c].astype(str)
    X_sub_cat[c] = X_sub_cat[c].astype(str)

for tr, va in kf.split(X):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr = y[tr]

    m_ridge = Pipeline([
        ('pre', pre),
        ('model', Ridge(alpha=3.0, random_state=42))
    ])
    m_ridge.fit(X_tr, y_tr)
    oof_ridge[va] = m_ridge.predict(X_va)
    pred_ridge += m_ridge.predict(X_sub) / kf.n_splits

    m_gbr = Pipeline([
        ('pre', pre),
        ('model', GradientBoostingRegressor(
            random_state=42,
            n_estimators=900,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=5,
            subsample=0.9
        ))
    ])
    m_gbr.fit(X_tr, y_tr)
    oof_gbr[va] = m_gbr.predict(X_va)
    pred_gbr += m_gbr.predict(X_sub) / kf.n_splits

    m_cat = CatBoostRegressor(
        iterations=900,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=False
    )
    m_cat.fit(
        X_cat_full.iloc[tr],
        y_tr,
        cat_features=cat_cols,
        eval_set=(X_cat_full.iloc[va], y[va]),
        use_best_model=True,
        early_stopping_rounds=80
    )
    oof_cat[va] = m_cat.predict(X_cat_full.iloc[va])
    pred_cat += m_cat.predict(X_sub_cat) / kf.n_splits

print(f'OOF Ridge R2: {r2_score(y, oof_ridge):.6f}')
print(f'OOF GBR R2:   {r2_score(y, oof_gbr):.6f}')
print(f'OOF Cat R2:   {r2_score(y, oof_cat):.6f}')

rng = np.random.default_rng(42)
best_r2 = -1.0
best_w = None
for _ in range(50000):
    w = rng.random(3)
    w = w / w.sum()
    blend_tmp = w[0] * oof_ridge + w[1] * oof_gbr + w[2] * oof_cat
    score = r2_score(y, blend_tmp)
    if score > best_r2:
        best_r2 = score
        best_w = w

w_ridge, w_gbr, w_cat = best_w
print(f'Best weights: ridge={w_ridge:.4f}, gbr={w_gbr:.4f}, cat={w_cat:.4f}')
oof_blend = w_ridge * oof_ridge + w_gbr * oof_gbr + w_cat * oof_cat
print(f'OOF Blend R2: {r2_score(y, oof_blend):.6f}')

pred_blend = w_ridge * pred_ridge + w_gbr * pred_gbr + w_cat * pred_cat
submission = pd.DataFrame({
    'id': test['id'],
    'rental_price': np.clip(pred_blend, 0.5, None)
})
submission.to_csv('submission', index=False)
print(f'Elapsed: {time.time() - start_time:.2f} sec')
submission.head()

OOF Ridge R2: 0.854982
OOF GBR R2:   0.860733
OOF Cat R2:   0.859918
Best weights: ridge=0.2596, gbr=0.4088, cat=0.3316
OOF Blend R2: 0.868009
Elapsed: 164.77 sec


,id,rental_price
0,0,7.856503
1,1,8.966074
2,2,5.160264
3,3,4.560682
4,4,9.274890


_Прогресс_

OOF Ridge R2: 0.854982

OOF GBR R2:   0.860733

OOF Cat R2:   0.859918

Best weights: ridge=0.2596, gbr=0.4088, cat=0.3316

OOF Blend R2: 0.868009

Elapsed: 165.83 sec